# DataGastro

## Radiografía del ecosistema gastronómico de la Ciudad de Buenos Aires

**Informe de presentación** — lectura ordenada del sector a partir de datos abiertos del Gobierno de la Ciudad de Buenos Aires (GCBA) y relevamientos trazables.

Esta notebook está pensada para mostrar. Cuenta la historia de principio a fin, con mapas y lecturas de gestión, y deja en claro qué se puede afirmar y qué no.

> *Generado de forma reproducible: corré `Kernel → Restart & Run All` y se reconstruye solo desde los datos del proyecto.*

## C?mo leer este informe

La regla de oro: **no se suman fuentes que miden cosas distintas.** Cada una responde una pregunta y vive separada de las dem?s.

- **Fuente 1: oferta gastron?mica registrada (F01)** ? gu?a oficial de oferta gastron?mica. Dice d?nde hay gastronom?a seg?n el registro, no si cada local sigue abierto.

- **Fuente 2: habilitaciones gastron?micas aprobadas (F02)** ? autorizaciones aprobadas por la Agencia Gubernamental de Control (AGC). Es la autorizaci?n formal para operar. **No son locales activos** ni cuentan cierres.

- **Fuente 3: espacios de ferias, mercados y Ferias Itinerantes de Abastecimiento Barrial (F03)** ? espacios reales de ferias, mercados y Ferias Itinerantes de Abastecimiento Barrial (FIAB). Contamos espacios, no puestos ni personas.

- **F04 / F05** ? eventos y programas relevados a mano, con la fuente anotada fila por fila. Inventario parcial, no universo completo.



Todo lo que sigue respeta esa separaci?n.

In [ ]:
from pathlib import Path
import json, unicodedata
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
from matplotlib.patches import Polygon as MplPoly
from matplotlib.collections import PatchCollection
import matplotlib.colors as mcolors
import matplotlib.cm as cmx

# Encontrar la raiz del proyecto, corra desde notebooks/ o desde la raiz
ROOT = Path.cwd()
while not (ROOT / "data" / "processed").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
PROC = ROOT / "data" / "processed"
ANALYTICS = ROOT / "data" / "analytics"
RAW = ROOT / "data" / "raw"
print("Raiz del proyecto:", ROOT)

def leer(path):
    p = Path(path)
    return pd.read_csv(p, dtype=str, keep_default_na=False) if p.exists() else pd.DataFrame()

def miles(n):
    return f"{int(n):,}".replace(",", ".")

def norm(s):
    return unicodedata.normalize("NFKD", str(s)).encode("ascii", "ignore").decode().upper().strip()

# Tablas base
resumen   = leer(ANALYTICS / "analytics_resumen_ejecutivo.csv")
hab_anio  = leer(ANALYTICS / "analytics_habilitaciones_por_anio.csv")
hab_cat   = leer(ANALYTICS / "analytics_habilitaciones_por_categoria.csv")
est_barrio= leer(ANALYTICS / "analytics_establecimientos_por_categoria_barrio.csv")
ubic      = leer(PROC / "dim_ubicacion.csv")
fact_est  = leer(PROC / "fact_establecimiento.csv")
fact_hab  = leer(PROC / "fact_habilitacion_gastronomica.csv")
fact_esp  = leer(PROC / "fact_espacio_feria_mercado.csv")
fact_ev   = leer(PROC / "fact_evento_gastronomico.csv")

def indicador(nombre):
    if resumen.empty or "indicador" not in resumen.columns: return 0
    fila = resumen[resumen["indicador"] == nombre]
    if fila.empty: return 0
    try: return int(float(fila.iloc[0]["valor"]))
    except Exception: return 0

# Coordenadas listas para mapear
ubic["lat"] = pd.to_numeric(ubic.get("latitud"), errors="coerce")
ubic["lon"] = pd.to_numeric(ubic.get("longitud"), errors="coerce")
geo = ubic[["id_ubicacion", "lat", "lon"]].copy()
geo["cg"] = ubic.get("calidad_geo", "")

def puntos(fact, calidad):
    if fact.empty: return pd.DataFrame(columns=["lat","lon"])
    m = fact[["id_ubicacion"]].merge(geo, on="id_ubicacion", how="left")
    return m[(m["cg"] == calidad) & m["lat"].notna()]

# Poligonos de barrios (contorno de referencia + coropletas)
def cargar_barrios():
    p = RAW / "geo_barrios.geojson"
    if not p.exists(): return []
    feats = json.loads(p.read_text(encoding="utf-8")).get("features", [])
    out = []
    for f in feats:
        g = f["geometry"]; coords = g["coordinates"]
        parts = coords if g["type"] == "Polygon" else [r for mp in coords for r in mp]
        rings = [[(pt[0], pt[1]) for pt in ring] for ring in parts]
        out.append({"nombre": f["properties"].get("nombre",""),
                    "area_km2": float(f["properties"].get("area_metro", 0) or 0)/1_000_000,
                    "rings": rings})
    return out
BARRIOS = cargar_barrios()

def contorno(ax):
    pc = [MplPoly(r, closed=True) for b in BARRIOS for r in b["rings"]]
    ax.add_collection(PatchCollection(pc, facecolor="#f4f4f4", edgecolor="#bcbcbc", linewidths=0.5))

def encuadrar(ax, x=(-58.54,-58.33), y=(-34.71,-34.53)):
    ax.set_xlim(*x); ax.set_ylim(*y); ax.set_aspect(1/0.82); ax.set_xticks([]); ax.set_yticks([])

print("Datos cargados. Todo listo.")

## 1. Los números, separados

Lo primero y más importante: cada fuente se informa por su lado. No existe —ni debe existir— un número único que sume todo.

In [ ]:
f01 = indicador("establecimientos_oferta_gastronomica_f01")
f02 = indicador("habilitaciones_gastronomicas_f02")
f02_geo = indicador("habilitaciones_f02_geocodificadas")
f03 = indicador("espacios_ferias_mercados_f03")
f04 = indicador("eventos_gastronomicos_reales_f04_aptos")

print("="*56)
print("  DATAGASTRO — CONTEOS PRINCIPALES (separados)")
print("="*56)
print(f"  F01  Oferta registrada en la guia oficial : {miles(f01):>8}")
print(f"  F02  Habilitaciones gastronomicas aprobadas: {miles(f02):>8}")
print(f"       de las cuales geocodificadas (USIG)   : {miles(f02_geo):>8}  ({f02_geo/f02*100:.0f}%)")
print(f"  F03  Espacios de ferias/mercados/FIAB      : {miles(f03):>8}")
print(f"  F04  Eventos verificados                   : {miles(f04):>8}")
print("="*56)
print("  Recordatorio: F02 no son locales activos.")

## 2. El mapa del ecosistema

Tres capas sobre la Ciudad: la oferta gastron?mica registrada (F01), los espacios de ferias, mercados y Ferias Itinerantes de Abastecimiento Barrial (F03) y ?la novedad? las habilitaciones gastron?micas aprobadas (F02) geocodificadas con el Sistema de Informaci?n Geogr?fica del GCBA (USIG). Cada una es un universo distinto.

In [ ]:
f01_pts = puntos(fact_est, "fuente_oficial")
f03_pts = puntos(fact_esp, "fuente_oficial")
f02_pts = puntos(fact_hab, "usig_exacta")

fig, (axa, axb) = plt.subplots(1, 2, figsize=(16, 8))
for ax in (axa, axb):
    contorno(ax); encuadrar(ax)

axa.scatter(f01_pts["lon"], f01_pts["lat"], s=7, c="#2b6cb0", alpha=0.55, linewidths=0,
            label=f"F01 oferta registrada ({miles(len(f01_pts))})")
axa.scatter(f03_pts["lon"], f03_pts["lat"], s=34, c="#1f8a4c", alpha=0.95, marker="^", linewidths=0,
            label=f"F03 ferias/mercados/FIAB ({miles(len(f03_pts))})")
axa.set_title("Oferta registrada y espacios públicos", fontsize=13)
axa.legend(loc="lower left", fontsize=9, framealpha=0.9)

axb.scatter(f02_pts["lon"], f02_pts["lat"], s=4, c="#ba4836", alpha=0.16, linewidths=0,
            label=f"F02 habilitaciones USIG ({miles(len(f02_pts))})")
axb.scatter(f01_pts["lon"], f01_pts["lat"], s=5, c="#2b6cb0", alpha=0.45, linewidths=0,
            label=f"F01 oferta ({miles(len(f01_pts))})")
axb.set_title("Con habilitaciones geocodificadas (F02 USIG)", fontsize=13)
axb.legend(loc="lower left", fontsize=9, framealpha=0.9)

fig.suptitle("El ecosistema gastronómico sobre el mapa de CABA", fontsize=15, y=0.97)
plt.tight_layout(rect=[0,0,1,0.95]); plt.show()

**Lectura.** La actividad se concentra con fuerza en el corredor norte y el centro, y se afina hacia el sur. La capa de habilitaciones geocodificadas (derecha) es la única que permite ver, calle por calle, dónde se aprueba actividad gastronómica formal en la Ciudad. **Son habilitaciones aprobadas, no locales activos.**

## 3. ¿Qué barrios concentran la oferta?

Acá hay un matiz que cambia la conclusión. En números absolutos manda Palermo; pero Palermo es enorme. Si miramos **densidad** (locales por km²), el verdadero núcleo es el microcentro y el casco histórico.

In [ ]:
# Conteo F01 por barrio
m = fact_est[["id_ubicacion"]].merge(ubic[["id_ubicacion","barrio"]], on="id_ubicacion", how="left")
m = m[~m["barrio"].isin(["No determinado",""])]
conteo = m["barrio"].map(norm).value_counts().to_dict()
area = {norm(b["nombre"]): b["area_km2"] for b in BARRIOS if b["area_km2"] > 0}
densidad = {n: conteo[n]/area[n] for n in conteo if area.get(n)}

def coropleta(ax, valores, titulo, sufijo="", topn=7, gamma=0.6):
    mx = max(valores.values()); ncol = mcolors.PowerNorm(gamma=gamma, vmin=0, vmax=mx); cmap = plt.colormaps["YlOrRd"]
    patches, colors, etiquetas = [], [], []
    for b in BARRIOS:
        n = norm(b["nombre"]); v = valores.get(n, 0)
        for r in b["rings"]:
            patches.append(MplPoly(r, closed=True)); colors.append(cmap(ncol(v)))
        if v > 0:
            pts = [p for r in b["rings"] for p in r]
            cx = sum(x for x,_ in pts)/len(pts); cy = sum(y for _,y in pts)/len(pts)
            etiquetas.append((cx, cy, b["nombre"], v))
    ax.add_collection(PatchCollection(patches, facecolor=colors, edgecolor="#888", linewidths=0.5))
    encuadrar(ax); ax.set_title(titulo, fontsize=13)
    top = {n for n,_ in sorted(valores.items(), key=lambda x:-x[1])[:topn]}
    for cx, cy, nombre, v in etiquetas:
        if norm(nombre) in top:
            txt = f"{nombre}\n{int(round(v))}{sufijo}"
            ax.annotate(txt, (cx, cy), ha="center", va="center", fontsize=8, fontweight="bold",
                        bbox=dict(boxstyle="round,pad=0.2", fc="white", ec="none", alpha=0.78))
    sm = cmx.ScalarMappable(norm=ncol, cmap=cmap); sm.set_array([])
    return sm

fig, (axa, axb) = plt.subplots(1, 2, figsize=(16, 8))
sm1 = coropleta(axa, conteo, "En números absolutos")
sm2 = coropleta(axb, densidad, "Por densidad (locales/km²)", sufijo="/km²")
fig.colorbar(sm1, ax=axa, fraction=0.04, pad=0.02).set_label("F01 (absoluto)", fontsize=9)
fig.colorbar(sm2, ax=axb, fraction=0.04, pad=0.02).set_label("F01 / km²", fontsize=9)
fig.suptitle("Oferta registrada F01 por barrio (registros, no locales activos)", fontsize=15, y=0.97)
plt.tight_layout(rect=[0,0,1,0.95]); plt.show()

print("Top absoluto :", ", ".join(f"{n.title()} {v}" for n,v in sorted(conteo.items(), key=lambda x:-x[1])[:5]))
print("Top densidad :", ", ".join(f"{n.title()} {v:.0f}/km²" for n,v in sorted(densidad.items(), key=lambda x:-x[1])[:5]))

**Lectura.** Palermo lidera en volumen, pero por su tamaño la oferta queda diluida. Por densidad, **San Nicolás multiplica por seis o siete la concentración de Palermo**: el corazón gastronómico por intensidad es el microcentro y el casco histórico (San Nicolás, Monserrat, San Telmo). Es la diferencia entre *dónde hay muchos* y *dónde están apretados*.

## 4. ¿Cuánto se abre por año?

Una habilitación aprobada es la autorización formal para operar un rubro en un domicilio: marca dónde el sector formal está invirtiendo. Mostramos solo los años comparables entre sí; los períodos con esquema distinto se aclaran aparte.

In [ ]:
serie = hab_anio.copy()
if "comparable_como_flujo_anual" in serie.columns:
    comp = serie[serie["comparable_como_flujo_anual"].astype(str) == "si"].copy()
    nocomp = serie[serie["comparable_como_flujo_anual"].astype(str) == "no"].copy()
else:
    comp, nocomp = serie, serie.iloc[0:0]
comp["n"] = pd.to_numeric(comp["cantidad_habilitaciones"], errors="coerce").fillna(0).astype(int)
comp = comp.sort_values("anio_fuente")

fig, ax = plt.subplots(figsize=(11, 5))
ax.bar(comp["anio_fuente"], comp["n"], color="#5c7c3f")
ax.set_title("Habilitaciones gastronómicas aprobadas por año (serie comparable)", fontsize=13)
ax.set_ylabel("Habilitaciones"); 
for x, y in zip(comp["anio_fuente"], comp["n"]):
    ax.annotate(miles(y), (x, y), ha="center", va="bottom", fontsize=9)
plt.tight_layout(); plt.show()

if not nocomp.empty:
    print("Períodos que se muestran aparte (no comparables como flujo anual):")
    for _, r in nocomp.iterrows():
        print(f"  - {r['anio_fuente']}: {miles(r['cantidad_habilitaciones'])} — {r.get('nota_serie','')}")

## 5. ¿Qué tipo de gastronomía se habilita?

La composición por rubro, con la clasificación corregida (matching por palabra completa y exclusiones trazables).

In [ ]:
cat = hab_cat.copy()
cat["n"] = pd.to_numeric(cat["cantidad_habilitaciones"], errors="coerce").fillna(0).astype(int)
cat = cat.sort_values("n", ascending=True)
fig, ax = plt.subplots(figsize=(10, 6))
ax.barh(cat["categoria_gastronomica_inferida"], cat["n"], color="#2f6f9f")
ax.set_title("Habilitaciones por categoría gastronómica", fontsize=13)
for y, v in enumerate(cat["n"]):
    ax.annotate(miles(v), (v, y), va="center", ha="left", fontsize=9)
plt.tight_layout(); plt.show()

## 6. Los espacios públicos: ferias, mercados y Ferias Itinerantes de Abastecimiento Barrial (FIAB)

La pata pública del ecosistema. Contamos espacios reales, no puestos ni personas.

In [ ]:
if not fact_esp.empty and "tipo_espacio" in fact_esp.columns:
    tipo = fact_esp["tipo_espacio"].value_counts()
    fig, ax = plt.subplots(figsize=(10, 4.5))
    ax.barh(tipo.index[::-1], tipo.values[::-1], color="#1f8a4c")
    ax.set_title("Espacios F03 por tipo", fontsize=13)
    for y, v in enumerate(tipo.values[::-1]):
        ax.annotate(miles(v), (v, y), va="center", ha="left", fontsize=9)
    plt.tight_layout(); plt.show()
    alim = (fact_esp.get("es_gastronomico","") == "si").sum()
    print(f"Total de espacios: {len(fact_esp)} — con perfil alimentario: {alim}")

## 7. Eventos y programas de la Ciudad

Relevamiento manual trazable: lo que la Ciudad impulsa o acompaña. Inventario documentado, **no universo completo ni medición de impacto.**

In [ ]:
ev_aptos = fact_ev[fact_ev.get("apto_dashboard","") == "si"] if not fact_ev.empty else pd.DataFrame()
print(f"Eventos relevados: {len(fact_ev)}  |  verificados (entran en métricas): {len(ev_aptos)}")
if not ev_aptos.empty and "tipo_evento" in ev_aptos.columns:
    print("\nEventos verificados por tipo:")
    for t, c in ev_aptos["tipo_evento"].value_counts().items():
        print(f"  - {t}: {c}")
print("\nNota: con 13 eventos verificados esto es evidencia de que existe un calendario sostenido, no una tendencia estadística.")

## 8. Qu? responde hoy y qu? todav?a no

**Responde:**

- D?nde se concentra la oferta gastron?mica registrada, en volumen y en densidad.

- C?mo evolucionan las habilitaciones aprobadas a?o a a?o y de qu? tipo son.

- D?nde, calle por calle, se aprueba actividad gastron?mica formal mediante habilitaciones gastron?micas aprobadas (F02) geocodificadas.

- Qu? espacios p?blicos, eventos y programas sostiene la Ciudad.



**Todav?a NO responde ?y conviene decirlo de frente:**

- Cu?ntos locales est?n activos hoy (no existe un padr?n vivo con bajas).

- Cu?ntos cerraron, ni cu?nto sobreviven.

- Empleo, ventas o impacto econ?mico del sector.

- Si un barrio est? saturado o vac?o en t?rminos reales (falta denominador de demanda).



Es un mapa del ecosistema **formal y registrado**, no una foto del mercado real. Esa honestidad es lo que lo hace confiable.

## 9. Hacia d?nde sigue

1. **Permisos de ?rea gastron?mica** (mesas y sillas en la v?a p?blica): se?al de actividad vigente, que cubre el punto d?bil de la oferta gastron?mica registrada (F01).

2. **An?lisis de redes** sobre los puntos geocodificados: detectar polos y corredores con centralidad y comunidades (PageRank, Laplaciano, modularidad).

3. **Export del informe** a HTML/PDF para circular despu?s de cada reuni?n.



### Sobre el tablero interactivo

Adem?s de este informe, el proyecto tiene un **dashboard en Streamlit** para explorar los mismos datos de forma interactiva: mapa con capas que se prenden y apagan, filtros por categor?a y comuna, y la coropleta de habilitaciones. Esta notebook es la pieza para **contar la historia**; el dashboard es para **explorar**. Se corre con:

```

python -m streamlit run dashboard/app.py

```



---

*DataGastro ? datos abiertos del Gobierno de la Ciudad de Buenos Aires (GCBA) y relevamientos trazables. Cada n?mero informa su fuente, su fecha y sus l?mites.*